# 📦 데이터 검증 및 번들 릴리스

이 노트북은 생성된 합성 데이터를 종합적으로 검증하고,
학습자용 **준비된 데이터 번들**로 패키징합니다.

## 검증 항목

| # | 검증 | 설명 |
|---|------|------|
| 1 | 종합 검증 | 스키마, 근거, 조건, 도구 호출 일관성 |
| 2 | 분할 누출 | 시나리오 패밀리 격리, 평가 오염 검사 |
| 3 | 백엔드 변환 | LoRA/OSFT 전용 포맷 내보내기 |
| 4 | 번들 빌드 | 매니페스트, 체크섬, 메타데이터 |
| 5 | 로더 검증 | 양쪽 백엔드의 로더/토크나이저 확인 |
| 6 | 데이터셋 카드 | 문서화 및 발행 |

## 번들 출력

```
tau-knowledge-v1/
├── manifest.json, checksums.sha256
├── canonical/{train,validation}.jsonl
├── training/{lora,osft}/{train,validation}.jsonl
├── kb/documents.jsonl
├── metadata/{provenance.jsonl, splits.json}
├── reports/{quality.json, quality.md, backend-validation.json}
├── DATASET_CARD.md, LICENSES/
└── configs/{lora.yaml, osft.yaml}
```

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys, os
from pathlib import Path

# 프로젝트 루트 탐색 (pyproject.toml 기준)
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

# 패키지 및 의존성 확인
_need_install = False
try:
    import rhoai_model_training_lab
except ImportError:
    _need_install = True

if _need_install:
    print("📦 패키지 설치 중...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", ".",
         "--extra-index-url", "https://pypi.org/simple/"],
        cwd=str(_project_root),
    )
    print("✅ 설치 완료")

# PROJECT_ROOT를 실제 프로젝트 경로로 강제 설정
import rhoai_model_training_lab.config as _cfg
_cfg.PROJECT_ROOT = _project_root
print(f"✅ 프로젝트 루트: {_project_root}")


In [7]:
"""설정 및 sdg_hub 원시 데이터 → CanonicalSample 변환."""

import os, sys, json, hashlib, shutil
from pathlib import Path
from collections import Counter

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, load_bundle_config, PROJECT_ROOT,
)

load_env()

sdg_config = load_yaml_config("configs/sdg.yaml")
prep_config = load_yaml_config("configs/data-preparation.yaml")
release_config = load_bundle_config()

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]
bundle_base = PROJECT_ROOT / release_config.get("bundle", {}).get(
    "base_path", "data/prepared/tau-knowledge-v1"
)

def rel(p: Path) -> str:
    """PROJECT_ROOT 기준 상대 경로 반환 (출력용)."""
    try:
        return str(p.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(p)

print("=" * 70)
print("🔄 Step 1: sdg_hub 원시 출력 → CanonicalSample 변환")
print("=" * 70)

from rhoai_model_training_lab.schemas.data import CanonicalSample, SampleType

all_samples: list[CanonicalSample] = []
convert_errors = 0

if canonical_path.exists():
    for cf in sorted(canonical_path.glob("*.jsonl")):
        if cf.name.startswith("converted"):
            continue  # 이전 변환 파일 건너뛰기

        with open(cf) as f:
            for lineno, line in enumerate(f, 1):
                if not line.strip():
                    continue
                try:
                    raw = json.loads(line)

                    # sdg_hub key_facts: question, response, id, key_fact, domain
                    question = raw.get("question", "")
                    response = raw.get("response", "")
                    doc_id = raw.get("id", "")

                    if not question or not response:
                        convert_errors += 1
                        continue

                    content_hash = hashlib.md5(
                        f"{question}{response}".encode()
                    ).hexdigest()[:8]
                    sample_id = f"kf-{doc_id}-{content_hash}" if doc_id else f"kf-{content_hash}"

                    sample = CanonicalSample(
                        sample_id=sample_id,
                        messages=[
                            {"role": "user", "content": question},
                            {"role": "assistant", "content": response},
                        ],
                        sample_type=SampleType.POLICY_QA,
                        source_doc_ids=[doc_id] if doc_id else None,
                        scenario_family=(
                            doc_id.rsplit("-", 1)[0] if doc_id and "-" in doc_id else doc_id
                        ),
                    )
                    all_samples.append(sample)
                except Exception as exc:
                    convert_errors += 1
                    if convert_errors <= 3:
                        print(f"  ⚠️ {cf.name}:{lineno}: {exc}")

    print(f"  ✅ 변환 성공: {len(all_samples)}개")
    if convert_errors:
        print(f"  ⚠️  변환 실패: {convert_errors}개")
else:
    print(f"  ❌ 경로 없음: {rel(canonical_path)}")
    print("     02_generate_synthetic.ipynb를 먼저 실행하세요.")


🔄 Step 1: sdg_hub 원시 출력 → CanonicalSample 변환
  ✅ 변환 성공: 1414개


In [8]:
"""중복 검사 및 분할 누출 방지."""

print("=" * 70)
print("🔒 Step 2: 중복 및 분할 누출 검사")
print("=" * 70)

if all_samples:
    # 시나리오 패밀리별 그룹핑
    family_groups: dict[str, list[str]] = {}
    for s in all_samples:
        fam = s.scenario_family or "unknown"
        family_groups.setdefault(fam, []).append(s.sample_id)

    sizes = [len(v) for v in family_groups.values()]
    print(f"  시나리오 패밀리: {len(family_groups)}개")
    print(f"  패밀리 크기: 평균 {sum(sizes)/len(sizes):.1f}, 최소 {min(sizes)}, 최대 {max(sizes)}")

    # 중복 ID
    all_ids = [s.sample_id for s in all_samples]
    dup_ids = len(all_ids) - len(set(all_ids))
    print(f"  중복 ID: {dup_ids}건 {'✅' if dup_ids == 0 else '⚠️ 제거 필요'}")

    # 내용 중복
    seen_hashes: set[str] = set()
    content_dupes = 0
    for s in all_samples:
        txt = " ".join(m.content or "" for m in s.messages if m.content)
        h = hashlib.md5(txt.encode()).hexdigest()
        if h in seen_hashes:
            content_dupes += 1
        seen_hashes.add(h)
    print(f"  내용 중복: {content_dupes}건 {'✅' if content_dupes == 0 else '⚠️'}")
else:
    family_groups = {}
    print("  ⚠️  검사할 데이터가 없습니다.")


🔒 Step 2: 중복 및 분할 누출 검사
  시나리오 패밀리: 19개
  패밀리 크기: 평균 74.4, 최소 35, 최대 119
  중복 ID: 2건 ⚠️ 제거 필요
  내용 중복: 2건 ⚠️


In [9]:
"""학습/검증 분할 → canonical + LoRA/OSFT 포맷 내보내기."""

import random

print("=" * 70)
print("📤 Step 3: 분할 및 백엔드별 내보내기")
print("=" * 70)

train_samples: list[CanonicalSample] = []
val_samples: list[CanonicalSample] = []

if all_samples and family_groups:
    train_ratio = prep_config["splits"]["train_ratio"]
    seed = prep_config["splits"]["seed"]
    rng = random.Random(seed)

    families = list(family_groups.keys())
    rng.shuffle(families)
    split_pt = int(len(families) * train_ratio)
    train_families = set(families[:split_pt])
    val_families = set(families[split_pt:])

    train_samples = [s for s in all_samples if (s.scenario_family or "unknown") in train_families]
    val_samples = [s for s in all_samples if (s.scenario_family or "unknown") in val_families]

    print(f"  학습: {len(train_samples)}개 ({len(train_families)} 패밀리)")
    print(f"  검증: {len(val_samples)}개 ({len(val_families)} 패밀리)")

    # ── Canonical splits ──
    for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
        out_dir = bundle_base / "canonical"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{split_name}.jsonl"
        with open(out_file, "w") as f:
            for s in samples:
                f.write(s.model_dump_json() + "\n")
        print(f"  ✅ {rel(out_file)}: {len(samples)}개")

    # ── LoRA / OSFT (messages-only) ──
    for backend in ["lora", "osft"]:
        for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
            out_dir = bundle_base / "training" / backend
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"{split_name}.jsonl"
            with open(out_file, "w") as f:
                for s in samples:
                    rec = {"messages": [m.model_dump(exclude_none=True) for m in s.messages]}
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"  ✅ {rel(out_file)}: {len(samples)}개")

    # ── KB 문서 복사 ──
    kb_src = PROJECT_ROOT / "data" / "sources" / "kb" / "documents.jsonl"
    if kb_src.exists():
        kb_dst = bundle_base / "kb"
        kb_dst.mkdir(parents=True, exist_ok=True)
        shutil.copy2(kb_src, kb_dst / "documents.jsonl")
        print(f"  ✅ {rel(kb_dst / 'documents.jsonl')}: KB 복사")
    else:
        print(f"  ⚠️  KB 소스 없음: {rel(kb_src)}")

    # ── Provenance 메타데이터 ──
    meta_dir = bundle_base / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)
    with open(meta_dir / "provenance.jsonl", "w") as f:
        for s in all_samples:
            prov = {
                "sample_id": s.sample_id,
                "source_doc_ids": s.source_doc_ids or [],
                "sample_type": s.sample_type.value,
            }
            f.write(json.dumps(prov, ensure_ascii=False) + "\n")
    print(f"  ✅ {rel(meta_dir / 'provenance.jsonl')}: 출처 메타데이터")

    print("\n  ℹ️  LoRA/OSFT 포맷: messages-only (assistant 응답만 학습 대상)")
else:
    print("  ⚠️  내보낼 데이터가 없습니다.")


📤 Step 3: 분할 및 백엔드별 내보내기
  학습: 1182개 (16 패밀리)
  검증: 232개 (3 패밀리)
  ✅ data/prepared/tau-knowledge-v1/canonical/train.jsonl: 1182개
  ✅ data/prepared/tau-knowledge-v1/canonical/validation.jsonl: 232개
  ✅ data/prepared/tau-knowledge-v1/training/lora/train.jsonl: 1182개
  ✅ data/prepared/tau-knowledge-v1/training/lora/validation.jsonl: 232개
  ✅ data/prepared/tau-knowledge-v1/training/osft/train.jsonl: 1182개
  ✅ data/prepared/tau-knowledge-v1/training/osft/validation.jsonl: 232개
  ✅ data/prepared/tau-knowledge-v1/kb/documents.jsonl: KB 복사
  ✅ data/prepared/tau-knowledge-v1/metadata/provenance.jsonl: 출처 메타데이터

  ℹ️  LoRA/OSFT 포맷: messages-only (assistant 응답만 학습 대상)


In [10]:
"""번들 빌드: 매니페스트, 체크섬, 분할 메타데이터."""

from rhoai_model_training_lab.data import compute_file_checksum
from rhoai_model_training_lab.schemas.data import BundleManifest, SplitInfo

print("=" * 70)
print("📦 Step 4: 번들 빌드")
print("=" * 70)

if all_samples and train_samples:
    model_id = release_config["model_profile"]["model_id"]
    model_revision = release_config["model_profile"]["model_revision"]
    type_dist = dict(Counter(s.sample_type.value for s in all_samples))

    manifest = BundleManifest(
        bundle_name=release_config["bundle"]["name"],
        bundle_version=release_config["bundle"]["version"],
        tau_version=os.environ.get("TAU_BENCH_VERSION", ""),
        tau_commit_sha=os.environ.get("TAU_BENCH_COMMIT_SHA", ""),
        model_id=model_id,
        model_revision=model_revision,
        tokenizer_id=release_config["model_profile"]["tokenizer_id"],
        canonical_train_count=len(train_samples),
        canonical_validation_count=len(val_samples),
        split_policy=prep_config["splits"]["method"],
        sample_type_distribution=type_dist,
    )

    manifest_path = bundle_base / "manifest.json"
    with open(manifest_path, "w") as f:
        f.write(manifest.model_dump_json(indent=2))
    print(f"  ✅ 매니페스트: {rel(manifest_path)}")

    # 체크섬
    checksum_path = bundle_base / "checksums.sha256"
    with open(checksum_path, "w") as f:
        for fp in sorted(bundle_base.rglob("*")):
            if fp.is_file() and fp.name != "checksums.sha256":
                h = compute_file_checksum(fp)
                f.write(f"{h}  {fp.relative_to(bundle_base)}\n")
    print(f"  ✅ 체크섬: {rel(checksum_path)}")

    # 분할 메타데이터
    split_info = SplitInfo(
        method=prep_config["splits"]["method"],
        seed=prep_config["splits"]["seed"],
        train_ids=[s.sample_id for s in train_samples],
        validation_ids=[s.sample_id for s in val_samples],
        train_count=len(train_samples),
        validation_count=len(val_samples),
        family_partition={
            f: "train" if f in train_families else "validation"
            for f in families
        },
    )
    meta_dir = bundle_base / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)
    with open(meta_dir / "splits.json", "w") as f:
        f.write(split_info.model_dump_json(indent=2))
    print(f"  ✅ 분할 메타: {rel(meta_dir / 'splits.json')}")

    print(f"\n  번들 경로: {rel(bundle_base)}")
else:
    print("  ⚠️  빌드할 데이터가 없습니다.")


📦 Step 4: 번들 빌드
  ✅ 매니페스트: data/prepared/tau-knowledge-v1/manifest.json
  ✅ 체크섬: data/prepared/tau-knowledge-v1/checksums.sha256
  ✅ 분할 메타: data/prepared/tau-knowledge-v1/metadata/splits.json

  번들 경로: data/prepared/tau-knowledge-v1


In [11]:
"""백엔드 로더 검증."""

from rhoai_model_training_lab.data import BundleManager, validate_prepared_dataset

print("=" * 70)
print("🔍 Step 5: 백엔드 로더 검증")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    validation = validate_prepared_dataset(bundle_base)

    if validation["valid"]:
        print("  ✅ 번들 검증 통과")
    else:
        print("  ❌ 번들 검증 실패")
        for err in validation.get("errors", []):
            print(f"    ❌ {err}")
    for warn in validation.get("warnings", []):
        print(f"    ⚠️  {warn}")

    mgr = BundleManager.load_bundle(bundle_base)

    for backend in ["lora", "osft"]:
        print(f"\n  --- {backend.upper()} 로더 ---")
        try:
            tr = mgr.get_training_samples(backend, "train")
            va = mgr.get_training_samples(backend, "validation")
            print(f"    학습: {len(tr)}개 ✅")
            print(f"    검증: {len(va)}개 ✅")
            if tr:
                assert "messages" in tr[0], "messages 필드 누락"
                print(f"    메시지 포맷 ✅")
        except Exception as exc:
            print(f"    ❌ {exc}")

    model_id = release_config["model_profile"]["model_id"]
    compat = mgr.validate_compatibility(model_id)
    print(f"\n  모델 호환성: {'✅' if not compat.errors else '❌'}")
    for err in compat.errors:
        print(f"    ❌ {err}")
else:
    print("  ⚠️  번들 없음 — Step 3, 4를 먼저 실행하세요.")


🔍 Step 5: 백엔드 로더 검증
  ✅ 번들 검증 통과
    ⚠️  Optional file missing: reports/quality.json
    ⚠️  Optional file missing: reports/quality.md
    ⚠️  Optional file missing: reports/backend-validation.json

  --- LORA 로더 ---
    학습: 1182개 ✅
    검증: 232개 ✅
    메시지 포맷 ✅

  --- OSFT 로더 ---
    학습: 1182개 ✅
    검증: 232개 ✅
    메시지 포맷 ✅

  모델 호환성: ✅


In [12]:
"""데이터셋 카드 생성 및 최종 요약."""

print("=" * 70)
print("📄 Step 6: 데이터셋 카드 및 발행")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    dataset_card = f"""# τ-Knowledge Banking Dataset Bundle

## Overview
- **Name**: {release_config['bundle']['name']}
- **Version**: {release_config['bundle']['version']}
- **Domain**: τ-Knowledge banking_knowledge
- **Model Target**: {release_config['model_profile']['model_id']}

## Contents
- Canonical training/validation samples
- LoRA and OSFT backend-specific exports
- KB document snapshot
- Provenance and split metadata

## Generation
Synthetic data generated using sdg_hub Key Facts flow.
See `manifest.json` for sample counts and type distribution.

## Split Methodology
Scenario family-based splitting: paraphrases, name/number substitutions,
and sibling examples remain in the same split.

## Redistribution
Check LICENSES/ directory for applicable conditions.
"""
    card_path = bundle_base / "DATASET_CARD.md"
    with open(card_path, "w") as f:
        f.write(dataset_card)
    print(f"  ✅ 데이터셋 카드: {rel(card_path)}")

    (bundle_base / "LICENSES").mkdir(exist_ok=True)

    # ── 최종 요약 ──
    print("\n" + "=" * 70)
    print("🎉 번들 준비 완료!")
    print("=" * 70)
    print(f"  번들 경로: {rel(bundle_base)}")
    if train_samples and val_samples:
        print(f"  학습 샘플: {len(train_samples)}")
        print(f"  검증 샘플: {len(val_samples)}")
    print(f"  대상 모델: {release_config['model_profile']['model_id']}")
    print("\n  다음 단계:")
    print("    📓 notebooks/03_lora_finetuning.ipynb — LoRA 파인튜닝")
    print("    📓 notebooks/04_osft_finetuning.ipynb — OSFT 파인튜닝")
else:
    print("  ⚠️  번들 없음 — 이전 Step을 먼저 실행하세요.")


📄 Step 6: 데이터셋 카드 및 발행
  ✅ 데이터셋 카드: data/prepared/tau-knowledge-v1/DATASET_CARD.md

🎉 번들 준비 완료!
  번들 경로: data/prepared/tau-knowledge-v1
  학습 샘플: 1182
  검증 샘플: 232
  대상 모델: Qwen/Qwen3-4B-Instruct-2507

  다음 단계:
    📓 notebooks/03_lora_finetuning.ipynb — LoRA 파인튜닝
    📓 notebooks/04_osft_finetuning.ipynb — OSFT 파인튜닝
